In [5]:
import os
import random
import sys
from pyspark.sql import SparkSession, functions as F

os.environ["JAVA_HOME"] = r"C:/Program Files/Eclipse Adoptium/jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = r"C:/hadoop"

In [6]:
spark = SparkSession.builder.master("local[*]").appName("wiki-explore").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [9]:
path = "synthetic_pageviews.txt"
random.seed(1)
titles = ["Main_Page", "Albert_Einstein", "Cloud_computing", "Apache_Spark", "India",
          "Special:Search", "Python_(programming_language)", "Chennai", "Tamil_Nadu", "Moon"]
with open(path, "w") as f:
    for _ in range(5000):
        f.write(f"{random.choice(['en','en','en','de','fr','ta','hi'])} "
                f"{random.choice(titles)} {random.randint(1, 500)} 0\n")
print("[info] no input file given -> generated SYNTHETIC data:", path, "\n")

[info] no input file given -> generated SYNTHETIC data: synthetic_pageviews.txt 



In [10]:
raw = spark.read.text(path)
df = (raw.select(F.split("value", " ").alias("p"))
         .where(F.size("p") >= 3)
         .select(F.col("p")[0].alias("project"),
                 F.col("p")[1].alias("title"),
                 F.col("p")[2].cast("long").alias("views")))
df.cache()

print("Records:", df.count())
df.printSchema()

Records: 5000
root
 |-- project: string (nullable = true)
 |-- title: string (nullable = true)
 |-- views: long (nullable = true)



In [11]:
print("\n--- Views per language project (top 10)")
df.groupBy("project").agg(F.sum("views").alias("total_views")) \
  .orderBy(F.desc("total_views")).show(10, truncate=False)


--- Views per language project (top 10)
+-------+-----------+
|project|total_views|
+-------+-----------+
|en     |526488     |
|hi     |184549     |
|ta     |183725     |
|de     |177154     |
|fr     |172231     |
+-------+-----------+



In [12]:
print("--- Top 10 English pages (excluding Main_Page and Special: pages)")
(df.where((F.col("project") == "en") & (F.col("title") != "Main_Page")
          & (~F.col("title").startswith("Special:")))
   .groupBy("title").agg(F.sum("views").alias("total_views"))
   .orderBy(F.desc("total_views")).show(10, truncate=False))

--- Top 10 English pages (excluding Main_Page and Special: pages)
+-----------------------------+-----------+
|title                        |total_views|
+-----------------------------+-----------+
|Apache_Spark                 |55531      |
|Cloud_computing              |55431      |
|Tamil_Nadu                   |53583      |
|Python_(programming_language)|53046      |
|India                        |52733      |
|Moon                         |50615      |
|Chennai                      |49034      |
|Albert_Einstein              |46304      |
+-----------------------------+-----------+



In [13]:
print("--- Distribution of views per record")
df.select(F.min("views").alias("min"), F.avg("views").alias("mean"),
          F.expr("percentile_approx(views, 0.5)").alias("median"),
          F.max("views").alias("max")).show()

spark.stop()

--- Distribution of views per record
+---+--------+------+---+
|min|    mean|median|max|
+---+--------+------+---+
|  1|248.8294|   248|500|
+---+--------+------+---+

